# Probabilistic Language Models Workshop

## Team Information
- **Team Number:** Group1
- **Team Members:**
    - Ce Chen | 9007166
    - Zhuoran Zhang | 9048508

## GitHub Link
https://github.com/chence/ProbabilisticLanguageModels.git

## Workshop Goal
This notebook demonstrates an NLP pipeline and four probabilistic language model methods using the Reuters corpus from NLTK.  
It includes preprocessing, model implementation, sentence probability examples, and short discussion notes.

## Talking Points
- We used the Reuters corpus as our text dataset.
- Our NLP pipeline includes tokenization and normalization.
- We implemented four methods: unigram, bigram, trigram, and bigram with Laplace smoothing.
- We also calculated sentence probability and log probability to show how these models are used in practice.
- The main difference between the models is how much context they use.
- Smoothing helps avoid zero probability for unseen word pairs.

## 1. Import Libraries
We start by importing the libraries needed for tokenization, counting, and n-gram generation.

In [1]:
import math
import re
import nltk
from collections import Counter
from nltk.corpus import reuters
from nltk.util import ngrams

nltk.download('reuters')

[nltk_data] Downloading package reuters to
[nltk_data]     /Users/chrischen/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


True

## 2. Load the Corpus
We use the Reuters corpus because it is a standard NLP dataset with many news articles.

### Talking Points
- Reuters is a widely used benchmark corpus in NLP.
- It gives us enough text to build simple language models.
- Using a public dataset makes our experiment reproducible.

In [2]:
file_ids = reuters.fileids()
raw_tokens = reuters.words(fileids=file_ids)

print("Number of documents:", len(file_ids))
print("Number of raw tokens:", len(raw_tokens))
print("First 30 raw tokens:", raw_tokens[:30])

Number of documents: 10788
Number of raw tokens: 1720901
First 30 raw tokens: ['ASIAN', 'EXPORTERS', 'FEAR', 'DAMAGE', 'FROM', 'U', '.', 'S', '.-', 'JAPAN', 'RIFT', 'Mounting', 'trade', 'friction', 'between', 'the', 'U', '.', 'S', '.', 'And', 'Japan', 'has', 'raised', 'fears', 'among', 'many', 'of', 'Asia', "'"]


## 3. Tokenization and Normalization
We normalize the text by converting words to lowercase and removing punctuation and non-alphabetic tokens.

### Talking Points
- Tokenization splits text into individual units such as words.
- Normalization makes the data cleaner and more consistent.
- Lowercasing helps us treat 'The' and 'the' as the same word.
- Removing punctuation reduces noise in the model.

In [3]:
def simple_tokenizer(tokens):
    return [str(token) for token in tokens]

def normalize(tokens):
    normalized = []
    for token in tokens:
        token = token.lower()
        if re.fullmatch(r"[a-z]+", token):
            normalized.append(token)
    return normalized

tokens = simple_tokenizer(raw_tokens)
normalized_tokens = normalize(tokens)

print("Number of normalized tokens:", len(normalized_tokens))
print("First 30 normalized tokens:", normalized_tokens[:30])

Number of normalized tokens: 1327140
First 30 normalized tokens: ['asian', 'exporters', 'fear', 'damage', 'from', 'u', 's', 'japan', 'rift', 'mounting', 'trade', 'friction', 'between', 'the', 'u', 's', 'and', 'japan', 'has', 'raised', 'fears', 'among', 'many', 'of', 'asia', 's', 'exporting', 'nations', 'that', 'the']


## 4. Method 1 – Unigram Model (MLE)

A unigram model assumes that each word is independent of the previous words.

$$
P(w) = \frac{\text{count}(w)}{N}
$$

where $N$ is the total number of normalized tokens.

### Talking Points
- The unigram model is the simplest probabilistic language model.
- It ignores word order and context.
- It only looks at how often each word appears in the corpus.
- This method is easy to build, but it is limited because it does not capture relationships between words.

In [4]:
unigram_counts = Counter(normalized_tokens)
N = len(normalized_tokens)
vocab = sorted(set(normalized_tokens))
vocab_size = len(vocab)

def unigram_prob(word):
    return unigram_counts[word] / N if N > 0 else 0

print("Vocabulary size:", vocab_size)
print("P('market') =", unigram_prob('market'))
print("P('the') =", unigram_prob('the'))

Vocabulary size: 29172
P('market') = 0.002118088521180885
P('the') = 0.0522002200220022


## 5. Method 2 – Bigram Model

A bigram model considers the previous word as context.

$$
P(w_2 \mid w_1) = \frac{\text{count}(w_1, w_2)}{\text{count}(w_1)}
$$

### Talking Points
- The bigram model uses one previous word as context.
- It captures basic word order information.
- This makes it more realistic than a unigram model.
- However, it can still fail when a word pair never appears in the training corpus.

In [5]:
bigrams = list(ngrams(normalized_tokens, 2))
bigram_counts = Counter(bigrams)

def bigram_prob(w1, w2):
    denominator = unigram_counts[w1]
    if denominator == 0:
        return 0
    return bigram_counts[(w1, w2)] / denominator

print("Number of bigrams:", len(bigrams))
print("P('market' | 'the') =", bigram_prob('the', 'market'))
print("P('said' | 'he') =", bigram_prob('he', 'said'))

Number of bigrams: 1327139
P('market' | 'the') = 0.00851653506935924
P('said' | 'he') = 0.48053691275167787


## 6. Method 3 – Trigram Model

A trigram model uses two previous words as context.

$$
P(w_3 \mid w_1, w_2) = \frac{\text{count}(w_1, w_2, w_3)}{\text{count}(w_1, w_2)}
$$

### Talking Points
- The trigram model uses more context than the bigram model.
- It can produce more precise local predictions.
- The trade-off is data sparsity, because longer sequences appear less often.
- This means many trigrams will have zero probability in a small or medium dataset.

In [6]:
trigrams = list(ngrams(normalized_tokens, 3))
trigram_counts = Counter(trigrams)

def trigram_prob(w1, w2, w3):
    denominator = bigram_counts[(w1, w2)]
    if denominator == 0:
        return 0
    return trigram_counts[(w1, w2, w3)] / denominator

print("Number of trigrams:", len(trigrams))
print("P('said' | 'the', 'company') =", trigram_prob('the', 'company', 'said'))
print("P('market' | 'in', 'the') =", trigram_prob('in', 'the', 'market'))

Number of trigrams: 1327138
P('said' | 'the', 'company') = 0.3772378516624041
P('market' | 'in', 'the') = 0.013252502467221204


## 7. Method 4 – Bigram Model with Laplace Smoothing

Laplace smoothing helps handle unseen word pairs.

$$
P(w_2 \mid w_1) = \frac{\text{count}(w_1, w_2) + 1}{\text{count}(w_1) + |V|}
$$

where $|V|$ is the vocabulary size.

### Talking Points
- Smoothing is important because real language data is sparse.
- Without smoothing, unseen bigrams get probability zero.
- With Laplace smoothing, every possible next word gets a small non-zero probability.
- This makes the model more robust for sentence probability calculations.

In [7]:
def bigram_prob_laplace(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + vocab_size)

print("Smoothed P('market' | 'the') =", bigram_prob_laplace('the', 'market'))
print("Smoothed P('banana' | 'the') =", bigram_prob_laplace('the', 'banana'))

Smoothed P('market' | 'the') = 0.0060031082083109024
Smoothed P('banana' | 'the') = 1.0157543499680038e-05


## 8. Sentence Probability

For a unigram model:

$$
P(w_1, w_2, \ldots, w_n) = \prod_{i=1}^{n} P(w_i)
$$

For a bigram model:

$$
P(w_1, w_2, \ldots, w_n) \approx P(w_1)\prod_{i=2}^{n} P(w_i \mid w_{i-1})
$$

### Talking Points
- Sentence probability is found by multiplying the probabilities of words or transitions.
- The final number is usually very small because many probabilities less than 1 are multiplied together.
- This is normal in language modeling.
- Bigram-based sentence probability is usually more informative because it uses local context.

In [8]:
def preprocess_sentence(sentence):
    return normalize(re.findall(r"[A-Za-z]+", sentence))

def sentence_prob_unigram(sentence):
    words = preprocess_sentence(sentence)
    prob = 1.0
    for word in words:
        prob *= unigram_prob(word)
    return words, prob

def sentence_prob_bigram_smoothed(sentence):
    words = preprocess_sentence(sentence)
    if not words:
        return words, 0.0
    prob = unigram_prob(words[0]) if unigram_prob(words[0]) > 0 else 1 / (N + vocab_size)
    for i in range(1, len(words)):
        prob *= bigram_prob_laplace(words[i-1], words[i])
    return words, prob

sample_sentence = "The market said the company will grow"
u_words, u_prob = sentence_prob_unigram(sample_sentence)
b_words, b_prob = sentence_prob_bigram_smoothed(sample_sentence)

print("Processed sentence:", u_words)
print("Unigram sentence probability:", u_prob)
print("Smoothed bigram sentence probability:", b_prob)

Processed sentence: ['the', 'market', 'said', 'the', 'company', 'will', 'grow']
Unigram sentence probability: 1.772059659270085e-16
Smoothed bigram sentence probability: 8.33325343607136e-16


## 9. Log Probability

Because sentence probabilities can become extremely small, log probability is often used.

$$
\log P(w_1, \ldots, w_n) = \sum_{i=1}^{n} \log P(w_i)
$$

### Talking Points
- Log probability avoids numerical underflow.
- Addition is easier and more stable than repeated multiplication.
- In practical NLP systems, log probabilities are commonly used.

In [9]:
def sentence_log_prob_unigram(sentence):
    words = preprocess_sentence(sentence)
    log_prob = 0.0
    for word in words:
        p = unigram_prob(word)
        if p == 0:
            return words, float('-inf')
        log_prob += math.log(p)
    return words, log_prob

log_words, log_prob = sentence_log_prob_unigram(sample_sentence)

print("Processed sentence:", log_words)
print("Unigram log probability:", log_prob)

Processed sentence: ['the', 'market', 'said', 'the', 'company', 'will', 'grow']
Unigram log probability: -36.269218968528946


## 10. Comparison of the Four Methods

### Observations
- **Unigram** is simple and fast, but it ignores context.
- **Bigram** is better because it considers one previous word.
- **Trigram** uses even more context, but data sparsity becomes a bigger issue.
- **Bigram with Laplace smoothing** is more practical because it can handle unseen word pairs.

### Talking Points
- As we add more context, the model becomes more expressive.
- At the same time, the number of unseen combinations increases.
- This is why smoothing is an important part of probabilistic language modeling.

## 11. Conclusion

In this workshop, we built an NLP pipeline and implemented four probabilistic language model methods: unigram, bigram, trigram, and smoothed bigram.  
These models show how probabilities can be used to represent language patterns.  
We also demonstrated sentence probability and log probability to explain how probabilistic language models work in practice.
